# W9L091 · Generics, Variance, and Type Projections

**CS3035**

We will use one pet application throughout: contests accept contestants, retailers return pets, and vets treat pets.

- How do you define your own generic classes and interfaces?
- How does polymorphism work with generic types?
- If you have a `MutableList<Animal>`, what happens if you try to assign a `MutableList<Dog>` to it?

    To answer these questions and more, we are going to create an application that uses generic types.




# Case Study: Pet Application
- To study generics, we are going to create an application that deals with pets.

- This app creates different pets, holds pet contests for them, and defines pet retailers that sell specific types of pets.

- We will use generics to ensure that each contest and retailer can work only with a specific type of pet.

Generic Types and the Pet Hierarchy

`Cat`, `Dog`, and `Fish` are subtypes of `Pet`. A value of any of these types can be used where a `Pet` is expected.


In [35]:
abstract class Pet(var name: String)
class Cat(name: String) : Pet(name)
class Dog(name: String) : Pet(name)
class Fish(name: String) : Pet(name)


In [36]:
val onePet: Pet = Cat("Katsu")
println(onePet.name)


Katsu


#### Abstract Classes (Quick Reminder)

- An **abstract class cannot be instantiated** directly.
- It is designed to be **inherited by subclasses**.
- It can contain:
  - **Abstract members** (no implementation)
  - **Concrete members** (with implementation)
- Abstract properties and functions **must be implemented** in subclasses using `override`.
- Abstract classes can have **constructors** to initialize shared properties.
- Use the `abstract` keyword to declare one.
- Abstract classes are **implicitly open** (no need for the `open` keyword).

**Key Idea**
> An abstract class defines a template.
> Subclasses provide the concrete implementation.

### Create a Contest Class

Read these parts of the class before looking at its implementation:

```kotlin
class Contest<T : Pet>
fun addScore(pet: T, score: Int = 0)
fun getWinners(): MutableSet<T>
```

- `T` is a type parameter. Choosing `Cat` makes the participant type `Cat` throughout that contest.
- `T : Pet` is an **upper bound**: valid type arguments include `Pet` and its subtypes. `Contest<String>` is rejected.
- The upper bound does not make `Contest<Cat>` a subtype of `Contest<Pet>`. That is a separate question about **variance**.


In [37]:
class Contest<T : Pet> {
    private val scores: MutableMap<T, Int> = mutableMapOf()

    fun addScore(pet: T, score: Int = 0) {
        if (score >= 0) scores[pet] = score
    }

    fun getWinners(): MutableSet<T> {
        val highScore = scores.values.maxOrNull() ?: return mutableSetOf()
        val winners: MutableSet<T> = mutableSetOf()
        for ((pet, score) in scores) {
            if (score == highScore) winners.add(pet)
        }
        return winners
    }
}


## 5–14 min · Invariance

### Let's Test the Contest Class

A `Contest<Cat>` accepts cats and returns cat winners. A `Contest<Pet>` can accept different kinds of pets.

**Predict before running:** Which call below must the compiler reject?


In [44]:
val catContest = Contest<Cat>()
catContest.addScore(Cat("Katsu"), 45)
catContest.addScore(Cat("Fuzz Lightyear"), 50)

catContest.addScore(Dog("Fido"), 23)

val petContest = Contest<Pet>()
petContest.addScore(Cat("Mochi"), 40)
petContest.addScore(Dog("Fido"), 23)
petContest.addScore(Fish("Nemo"), 56)


org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[44], line 5, column 21: Type mismatch: inferred type is Line_36_jupyter.Dog but Line_36_jupyter.Cat was expected

> #### Scope Function
> We use the `forEach()` function to iterate over and print the items in the `catsWinner` set.

In [41]:
catContest.getWinners().forEach{
    println("Cat: ${it.name}")
}

Cat: Fuzz Lightyear


In [43]:
petContest.getWinners().forEach{
    println("Pet: ${it.name}")
}

Pet: Nemo



> Notes
* The compiler prevents you from adding non-`Cat` objects, so that line will not compile.

* It is important to note that a `Contest<Pet>` can accept any subtype of `Pet`.


### Exercise 1 · Explain the Rejected Assignment (3 minutes)

Read this hypothetical sequence:

```kotlin
val cats: Contest<Cat> = Contest<Cat>()
val pets: Contest<Pet> = cats
pets.addScore(Dog("Fido"), 10)
```

1. Which line is rejected?
2. If that assignment were allowed, what could go wrong?
3. Would the reverse assignment, from `Contest<Pet>` to `Contest<Cat>`, be safe?


In [49]:
val catsForDemo = Contest<Cat>()
val petsForDemo: Contest<Pet> = catsForDemo
petsForDemo.addScore(Dog("Fido"), 10)


org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[49], line 2, column 33: Type mismatch: inferred type is Line_38_jupyter.Contest<Line_36_jupyter.Cat> but Line_38_jupyter.Contest<Line_36_jupyter.Pet> was expected

<details>
<summary>Exercise 1 answer · Reveal after discussion</summary>

The second line is rejected. The two references would point to the same contest, so calling `addScore` through `pets` could insert a dog into a contest promised to contain cats.

The reverse assignment is unsafe too: a `Contest<Pet>` might already contain a fish, and `getWinners()` would no longer promise cat winners.

`Contest<T>` is **invariant**: even though `Cat` is a subtype of `Pet`, neither `Contest<Cat>` nor `Contest<Pet>` is a subtype of the other. This API both accepts and exposes contestants of type `T`.

Accepting a cat as a participant in `Contest<Pet>` is different from assigning an entire `Contest<Cat>` to a `Contest<Pet>` variable.

</details>


##  Covariance: A Retailer Produces Pets
### The Retailer Hierarchy
Retailers can sell different types of `Pet`. To model this, we define a generic `Retailer` interface with a `sell` function, and three concrete classes: `CatRetailer`, `DogRetailer`, and `FishRetailer` that implement the interface.

Each retailer should only be able to sell a specific type of pet. For example, a `CatRetailer` can sell only `Cat` objects. To enforce this restriction, we use generics to specify the exact type each retailer workswith, ensuring type safety at compile time.



In [50]:
interface Retailer<T> {
    fun sell(): T
}

The `CatRetailer` class implements the `Retailer` interface so that it works specifically with `Cat` objects.

This means that the `sell()` function must return a `Cat`.|

In [52]:
class CatRetailer: Retailer<Cat> {
    override fun sell(): Cat {
        print("Sell Cat")
        return Cat("")
    }
}

What happens if you try to use something other than a `Cat` in `CatRetailer`?

The compiler will produce a type error. Since `CatRetailer` is defined to work only with `Cat` objects, attempting to use any other type violates the generic type constraint and will not compile. This ensures type safety at compile time.

In [53]:
class CatRetailer: Retailer<Cat> {
    override fun sell(): Dog {
        return Dog("")
    }
}

org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[53], line 2, column 26: Return type of 'sell' is not a subtype of the return type of the overridden member 'public abstract fun sell(): Line_36_jupyter.Cat defined in Line_51_jupyter.Retailer'

In [55]:
class DogRetailer:  Retailer<Dog> {
    override fun sell(): Dog {
        print("Sell Dog")
        return Dog("")
    }
}

## What About Polymorphism?

Since `CatRetailer`, `DogRetailer`, and `FishRetailer` implement the `Retailer` interface, we should be able to create a variable of type `Retailer` and assign one of its concrete implementations to it.

In [56]:
val dogRetailer: Retailer<Dog> = DogRetailer()
val catRetailer: Retailer<Cat> = CatRetailer()

In [ ]:
val catList: List<Cat> = listOf(Cat("Mochi"), Cat("Katsu"))
val petList: List<Pet> = catList
println(petList[0].name)


What Happens if We Assign `CatRetailer` to `Retailer<Pet>`?

In [59]:
// val petRetailer: Retailer<Pet> = CatRetailer()

* Even though `CatRetailer` is a subtype of `Retailer`, and `Cat` is a subtype of `Pet`, Kotlin does not allow assigning a `Retailer<Cat>` to a `Retailer<Pet>`.

* A variable of type `Retailer<Pet>` can only reference an object whose generic type parameter is exactly `Pet`, not a subtype like `Cat`.

* This may seem to contradict polymorphism, which normally allows subtypes to be used interchangeably. However, generic types in Kotlin are invariant by default, meaning that `Retailer<Cat>` is not considered a subtype of `Retailer<Pet>`.


## How Can We Fix This?
The good news is that we can adjust the generic type parameter in the `Retailer` interface to control which assignments are allowed.

### Solution: Use `out` (Covariance)

We can use the `out` keyword (also called **covariance**) to allow a `Retailer<Cat>` to be treated as a `Retailer<Pet>`. This lets us use a generic subtype in place of a generic supertype while still preserving type safety.

### Covariance Definition

        If a generic type is **covariant**, then:

- whenever `Cat <: Pet`, we also have `Retailer<Cat> <: Retailer<Pet>`

        In other words, covariance allows you to use a **subtype** where a **supertype** is expected for the same generic type.

In [60]:
interface Retailer<out T> {
    fun sell(): T
}

class CatRetailer: Retailer<Cat> {
    override fun sell(): Cat {
        print("Sell Cat")
        return Cat("")
    }
}

In [61]:
val petRetailer: Retailer<Pet> = CatRetailer()

>**Note:** A generic type parameter marked with `out` can be used only in an **out-position**, such as a function return type. It cannot be used in an **in-position**, meaning it cannot be used as a function parameter.

In [62]:
val catList: List<Cat> = listOf(Cat("c1"), Cat("c2"))
val petList: List<Pet> = catList

This works because some of Kotlin’s built-in collections use the `out` modifier. For example, the `List` interface is defined as:

``` Kotlin
public interface List<out T> ... { ... }
```


This means that `List` is covariant in `T`, allowing it to **produce** values of type `T` (e.g., through return types), but not **consume** them (e.g., as function parameters).

## Our Pet Application So Far

In [64]:
abstract class Pet(var name: String)
class Cat(name: String): Pet(name)
class Dog(name: String): Pet(name)
class Fish(name: String): Pet(name)

class Contest<T: Pet> {
    val scores: MutableMap<T, Int> = mutableMapOf()
    fun addScore(t: T, score: Int = 0) {
        if (score >=0) scores.put(t, score)
    }
    //The getWinners function need to return the contestants with the highest score.
    fun getWinners(): MutableSet<T>{
        val highScore = scores.values.max()
        val winners: MutableSet<T> = mutableSetOf()

        for ((t, score) in scores) {
            if (score == highScore) winners.add(t) // add any contestant with the highest score to a Mutable set
        }
        return winners
    }
}

interface Retailer<out T> {
    fun sell(): T
}
class CatRetailer: Retailer<Cat> {
    override fun sell(): Cat {
        print("Sell Cat")
        return Cat("")
    }
}

class DogRetailer:  Retailer<Dog> {
    override fun sell(): Dog {
        print("Sell Dog")
        return Dog("")
    }
}
class FishRetailer:  Retailer<Fish> {
    override fun sell(): Fish {
        print("Sell Fish")
        return Fish("")
    }
}

In [66]:
//Test
val catFuzz = Cat("Fuzz Lightyear")
val catKatsu = Cat("Katsu")
val fishFinny = Fish("Finny McGraw")
val nemo = Fish("Nemo")

val catContest = Contest<Cat>() // this create a Contest taht will accept Cats
catContest.addScore(catFuzz, 50)
catContest.addScore(catKatsu, 45)

val topCat = catContest.getWinners().first()
print("Cat contest winner is ${topCat.name}")


val petContest = Contest<Pet>()
petContest.addScore(catFuzz, 56)
petContest.addScore(fishFinny, 56)
petContest.addScore(nemo, 56)

val topPet = petContest.getWinners().first()
print("Pet contest winner is ${topPet.name}")

//Create some Retailer objects
val dogRetailer: Retailer<Dog> = DogRetailer()
val catRetailer: Retailer<Cat> = CatRetailer()
val petRetailer: Retailer<Pet> = CatRetailer()
petRetailer.sell()

Cat contest winner is Fuzz LightyearPet contest winner is Fuzz LightyearSell Cat

Line_65_jupyter$Cat@4b911d7

In [67]:
val petWinners = petContest.getWinners()
petWinners.forEach{
    println("Pet: ${it.name}")
}

Pet: Fuzz Lightyear
Pet: Finny McGraw
Pet: Nemo


## Contravariance: A Vet Accepts Pets

### Vet Class

* We want to assign a vet to each contest to handle medical emergencies for any contestant.

* Since vets may specialize in treating different types of pets, we will create a `Vet` class with a generic type `T`. This class will include a `treat` function that accepts a parameter of type `T`.

* To ensure type safety, we will restrict `T` to be a subtype of `Pet`. This guarantees that a vet can treat only valid pet types and prevents unintended cases, such as creating

For example
A cat contest needs a vet who can treat every cat in the contest. A vet who treats all pets can do that job.


In [68]:
class Vet<T: Pet> {
    fun treat(t: T){
        println("Treat Pet ${t.name}")
    }
}

We are adding a `Vet<T>` parameter to the `Contest` constructor so that you cannot create a `Contest` without assigning a `Vet` to it.

In [70]:
class Contest<T: Pet> (var vet: Vet<T>) {
    val scores: MutableMap<T, Int> = mutableMapOf()
    fun addScore(t: T, score: Int = 0) {
        if (score >=0) scores.put(t, score)
    }
    //The getWinners function need to return the contestants with the highest score.
    fun getWinners(): MutableSet<T>{
        val highScore = scores.values.max()
        val winners: MutableSet<T> = mutableSetOf()
        for ((t, score) in scores) {
            if (score == highScore) winners.add(t) // add any contestant with the highest score to a Mutable set
        }
        return winners
    }
}

Let's test our `Vet` class by creating instances of vets for different types of pets. This will help us verify that the generic type constraint works as expected.

In [71]:
val catVet = Vet<Cat>()
val fishvet = Vet<Fish>()
val petVet = Vet<Pet>()

It is important to note that a `Vet<Cat>` can treat only `Cat` objects, while a `Vet<Pet>` can treat any subtype of `Pet`, such as `Cat` or `Fish`.

However, a `Vet<Cat>` cannot treat anything that is not a `Cat`, such as a `Fish`.

In [73]:
catVet.treat(Cat("cat1"))
petVet.treat(Cat("cat1"))
petVet.treat(Fish("nemo"))
//catVet.treat(Fish("nemo"))

Treat Pet cat1
Treat Pet cat1
Treat Pet nemo


In [75]:
val catContest = Contest<Cat>(catVet)
val petContest = Contest<Pet>(petVet)
//val catContest = Contest<Cat>(petVet)

A `Vet<Pet>` can treat all types of `Pet`, including `Cat`. However, we cannot assign a `Vet<Pet>` to a `Contest<Cat>`, because the code will not compile.


## What Should We Do in This Situation?

### Solution: Make the Generic Type Contravariant

In this case, we can solve the problem by prefixing the generic type parameter of the `Vet` class with the `in` keyword.

The `in` modifier (contravariance) is the opposite of `out`. It allows a generic type that accepts a supertype to be used where a subtype is expected.

In [76]:
class Vet<in T : Pet> {
    fun treat(pet: T) {
        println("Treating pet: ${pet.name}")
    }
}

class Contest<T : Pet>(val vet: Vet<T>) {
    val scores: MutableMap<T, Int> = mutableMapOf()

    fun addScore(pet: T, score: Int = 0) {
        if (score >= 0) scores[pet] = score
    }

    // The getWinners function needs to return the contestants with the highest score.
    fun getWinners(): MutableSet<T> {
        val highScore = scores.values.maxOrNull() ?: return mutableSetOf()
        val winners: MutableSet<T> = mutableSetOf()

        for ((pet, score) in scores) {
            if (score == highScore) winners.add(pet) // Add any contestant with the highest score to the set
        }
        return winners
    }
}


If a generic type is **contravariant**, it means that you can use a **supertype** in place of a **subtype**.

        In our code, this means that we can use a `Vet<Pet>` in place of a `Vet<Cat>`.


In [77]:
val petVet = Vet<Pet>()
val catContest = Contest<Cat>(petVet)

## Should a `Contest<Cat>` Always Require a `Vet<Cat>`?

It depends.

### Scenario

Suppose we want to allow a `Vet<Pet>` to be used in place of a `Vet<Cat>`, but only when it is passed into the `Contest<Cat>` constructor.

In [78]:
// Remove the `in` modifier from the Vet class.
class Vet<T : Pet> {
    fun treat(pet: T) {
        println("Treating pet: ${pet.name}")
    }
}

// Add `in` to the Contest constructor instead.
// This makes the vet parameter contravariant only at the point where it is passed in.
class Contest<T : Pet>(var vet: Vet<in T>) {
    val scores: MutableMap<T, Int> = mutableMapOf()

    fun addScore(pet: T, score: Int = 0) {
        if (score >= 0) scores[pet] = score
    }

    // The getWinners function needs to return the contestants with the highest score.
    fun getWinners(): MutableSet<T> {
        val highScore = scores.values.maxOrNull() ?: return mutableSetOf()
        val winners: MutableSet<T> = mutableSetOf()

        for ((pet, score) in scores) {
            if (score == highScore) winners.add(pet) // Add any contestant with the highest score to the set
        }
        return winners
    }
}

In [79]:
val petVet = Vet<Pet>()            // A vet that can treat any Pet
val catContest = Contest<Cat>(petVet)  // Allowed because Vet<Pet> can act as Vet<in Cat>


## References

- Aigner, S., Elizarov, R., Isakova, S., & Jemerov, D. (2024). *Kotlin in Action*. Simon & Schuster.

- Griffiths, D., & Griffiths, D. (2019). *Head First Kotlin: A Brain-Friendly Guide*. O’Reilly Media.

- Kotlin Documentation. (n.d.). *Kotlin Tour*. https://kotlinlang.org/docs/kotlin-tour-welcome.html
